# 🌸 PyTorch Basics — Classifying Iris Flowers
### *Your First Neural Network in PyTorch, Step by Step*

---

## 🎯 What This Notebook Teaches

We will build a neural network that looks at a flower's measurements and predicts its species.  
Simple goal — but every single step maps directly to real-world deep learning projects.

| Step | What We Do | PyTorch Concept |
|------|-----------|----------------|
| 1 | Setup imports + device | `torch`, `nn`, `optim` |
| 2 | Load and understand data | NumPy → Tensor |
| 3 | Prepare data for training | `StandardScaler`, `train_test_split` |
| 4 | Build the neural network | `nn.Module` |
| 5 | Train the model | 5-Step Training Loop |
| 6 | Evaluate performance | `model.eval()`, `no_grad()` |
| 7 | Predict on new data | Inference pattern |
| 8 | Save and load the model | `state_dict` |

---

## 🌸 The Problem

```
       sepal length
       sepal width       ┌──────────────────┐    Setosa     (class 0)
    ─────────────────▶   │  Neural Network  │ ─▶ Versicolor (class 1)
       petal length       └──────────────────┘    Virginica  (class 2)
       petal width

       4 numbers in                               1 prediction out
```

- **150 samples**, **4 features**, **3 classes** — 50 flowers per species
- Small enough to train in seconds, real enough to teach every concept

---


---
## ✅ Step 1 — Imports and Setup

**What:** Import every library we need, set a seed, choose a device.

**Why set a seed?**  
Neural networks start with *random* weights. Without a fixed seed you get different results every run — impossible to debug or compare. `torch.manual_seed(42)` locks in the randomness so your run matches mine.

**Why choose a device?**  
PyTorch can run on CPU or GPU. The same code works on both — you just pick `device` once and everything moves to it. Iris trains fine on CPU, but writing this pattern now means you never forget it on a real project.

---
⚠️ **Common Mistake:** Forgetting to move your model **and** your data to the same device.  
Model on GPU + data on CPU → `RuntimeError: Expected all tensors to be on the same device`.  
**Rule: always call `.to(device)` on both model and input tensors.**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import seaborn as sns

# ── Reproducibility ───────────────────────────────────────────────────────────
# Fixes randomness so every run gives the same result.
# Always set this at the very top of your notebook.
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# ── Device ────────────────────────────────────────────────────────────────────
# PyTorch checks if a GPU is available; falls back to CPU if not.
# Move both model and tensors to `device` and the same code runs on either.
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"PyTorch version : {torch.__version__}")
print(f"Running on      : {device.upper()}")

---
## ✅ Step 2 — Load and Explore the Data

**What:** Load the Iris dataset and look at it before doing anything.

**Why explore first?**  
Never train a model blindly. A 2-minute look tells you:
- How many samples and features → determines model size
- Are classes balanced? → if 90% is one class, accuracy is a useless metric
- What are the feature ranges? → tells you whether scaling is needed (almost always yes)

---
⚠️ **Common Mistake:** Skipping data exploration. The most common cause of "my model doesn't learn" is a data problem — wrong shapes, unbalanced classes, unscaled features — not a model problem.

In [ ]:
# ── Load ──────────────────────────────────────────────────────────────────────
iris         = load_iris()
X            = iris.data          # numpy array (150, 4)
y            = iris.target        # numpy array (150,)  values: 0, 1, 2
CLASS_NAMES  = iris.target_names  # ['setosa', 'versicolor', 'virginica']

# ── Basic facts ───────────────────────────────────────────────────────────────
print(f"X shape : {X.shape}  →  {X.shape[0]} samples, {X.shape[1]} features")
print(f"Classes : {list(CLASS_NAMES)}")
print()

# ── Class balance ─────────────────────────────────────────────────────────────
# This dataset is perfectly balanced (50 per class).
# In real projects, imbalance is the norm — always check!
print("Class Distribution:")
for i, name in enumerate(CLASS_NAMES):
    print(f"  Class {i} ({name}) : {np.sum(y == i)} samples")
print()

# ── Feature ranges ────────────────────────────────────────────────────────────
# Sepal length: 4-8 cm. Petal width: 0-2.5 cm.
# These different scales will hurt gradient descent → we must normalize.
print(f"{'Feature':<25} {'Min':>5}  {'Max':>5}")
print("-" * 38)
for i, name in enumerate(iris.feature_names):
    print(f"{name:<25} {X[:,i].min():>5.1f}  {X[:,i].max():>5.1f}")

In [ ]:
# ── Quick scatter: are the classes visually separable? ────────────────────────
# Petal length vs petal width are the most discriminative features.
# If the classes cluster here, a neural network should handle them easily.

COLORS = ['#e74c3c', '#3498db', '#2ecc71']
fig, ax = plt.subplots(figsize=(7, 5))

for i, name in enumerate(CLASS_NAMES):
    mask = (y == i)
    ax.scatter(X[mask, 2], X[mask, 3], c=COLORS[i], label=name, s=60, alpha=0.8)

ax.set_xlabel('Petal Length (cm)')
ax.set_ylabel('Petal Width (cm)')
ax.set_title('Iris — Petal Features by Class')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# What to observe:
# Setosa (red)               → well separated
# Versicolor / Virginica     → some overlap → the network needs to work here

---
## ✅ Step 3 — Prepare Data for Training

Three things, in a specific order that matters:

### 3a — Split (train / test)
Hold out 20% of data the model will **never see during training**. This measures real generalization.

### 3b — Scale (StandardScaler)
Gradient descent treats all features as equally important. If one feature is 10× larger, its gradient is 10× larger and the optimization landscape is skewed — training is slow and unstable.

$$x_{\text{scaled}} = \frac{x - \mu}{\sigma} \quad \rightarrow \quad \text{mean} \approx 0, \text{ std} \approx 1$$

### 3c — Convert to Tensors
PyTorch only understands tensors — not numpy arrays.

---
⚠️ **Common Mistake #1 — Scaling before splitting (Data Leakage):**  
If you `fit_transform` on the full dataset, the scaler learns the test set's mean and std. Your model indirectly "sees" test distribution info → inflated metrics. **Always split first, then fit the scaler on training data only.**

⚠️ **Common Mistake #2 — Wrong tensor dtype:**  
- Features `X` → `torch.float32` — matches `nn.Linear` weight type  
- Labels `y` → `torch.long` (int64) — `CrossEntropyLoss` requires this  
Getting this wrong gives a cryptic dtype error deep in the loss function.

In [ ]:
# ── 3a: Split BEFORE scaling ──────────────────────────────────────────────────
# stratify=y → each split keeps the same class ratio (50/50/50)
# Without stratify, random chance could put all Setosa in the test set!
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f"Train : {X_train.shape}  ({len(X_train)} samples)")
print(f"Test  : {X_test.shape}   ({len(X_test)} samples)")
print()

# ── 3b: Scale using training statistics ONLY ──────────────────────────────────
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)  # Learns μ and σ from training data
X_test  = scaler.transform(X_test)       # Applies training μ/σ to test — no leakage!

print(f"After scaling — Train mean : {X_train.mean(axis=0).round(2)}  (should be ≈ 0)")
print(f"After scaling — Train std  : {X_train.std(axis=0).round(2)}   (should be ≈ 1)")
print()

# ── 3c: Convert to tensors ────────────────────────────────────────────────────
X_train_t = torch.tensor(X_train, dtype=torch.float32)  # float32 for features
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)     # int64 for class labels!
y_test_t  = torch.tensor(y_test,  dtype=torch.long)

print(f"X_train_t : {X_train_t.shape}  dtype={X_train_t.dtype}")
print(f"y_train_t : {y_train_t.shape}   dtype={y_train_t.dtype}  ← must be long for CrossEntropyLoss")

---
## ✅ Step 4 — Build the Neural Network

**What:** Define our model by subclassing `nn.Module`.

**Why `nn.Module`?**  
It is the base class for *every* neural network in PyTorch. Inheriting from it gives you automatic parameter tracking, `.to(device)`, `.train()` / `.eval()` modes, and `state_dict` for saving — for free.

**Two mandatory methods — always:**
```
__init__   →  define your layers  (the LEGO bricks)
forward    →  define data flow    (how to assemble them)
```

**Our architecture:**
```
Input (4)  →  Linear(4→16)  →  ReLU  →  Linear(16→3)  →  3 raw scores
```
Two layers is plenty for Iris. Match model complexity to the problem.

---
⚠️ **Common Mistake #1 — Forgetting `super().__init__()`:**  
Without it, PyTorch's internal state is never set up. Calling `.parameters()` or `.to(device)` will crash with a confusing `AttributeError`. Make it the very first line of `__init__`, always.

⚠️ **Common Mistake #2 — Adding Softmax to the output layer:**  
`CrossEntropyLoss` applies Softmax *internally*. If you also add `nn.Softmax` in `forward()`, the model trains on *double-softmax* — and learns incorrectly with no error message. **Return raw scores (logits) from `forward()`.**

In [ ]:
class IrisNet(nn.Module):
    """
    A simple 2-layer neural network for 3-class Iris classification.

    Architecture:  Linear(4→16) → ReLU → Linear(16→3)
    Output: raw logits (3 scores, one per class). No Softmax here.
    """

    def __init__(self):
        super().__init__()  # ALWAYS first — initialises nn.Module machinery

        self.layer1 = nn.Linear(4, 16)
        # nn.Linear(in_features, out_features)
        # Creates a weight matrix (16×4) and bias (16,) automatically
        # and registers them as learnable Parameters.

        self.relu = nn.ReLU()
        # Activation function: f(x) = max(0, x)
        # WHY: Without activations, stacking Linear layers is
        # mathematically equivalent to ONE Linear layer — the network
        # can only draw straight lines. ReLU adds non-linearity,
        # enabling the network to learn curved decision boundaries.

        self.layer2 = nn.Linear(16, 3)
        # Maps 16 hidden neurons → 3 output scores (one per class)
        # No Softmax! CrossEntropyLoss handles that internally.

    def forward(self, x):
        """
        How data flows through the network.
        Called automatically when you do: model(x)

        x in  : (batch_size, 4)
        x out : (batch_size, 3)  — 3 raw class scores
        """
        x = self.layer1(x)   # (batch, 4)  → (batch, 16)
        x = self.relu(x)     # (batch, 16) → (batch, 16)  same shape
        x = self.layer2(x)   # (batch, 16) → (batch, 3)
        return x              # Raw logits — not probabilities


# ── Create model and move to device ──────────────────────────────────────────
model = IrisNet().to(device)
print(model)
print()

# Count total parameters — understand what your model actually contains
total = sum(p.numel() for p in model.parameters())
print(f"Total trainable parameters: {total}")
# layer1 : 4×16 weights + 16 biases = 80
# layer2 : 16×3 weights +  3 biases = 51
# Total  : 131  — a tiny model, perfect for Iris

# ── Forward pass sanity check ────────────────────────────────────────────────
# Before training, verify the shapes work end-to-end.
with torch.no_grad():
    dummy_out = model(torch.randn(5, 4).to(device))
print(f"\nDummy forward pass: input (5, 4) → output {tuple(dummy_out.shape)}  ✅")

---
## ✅ Step 5 — Train the Model

**What:** Run the 5-Step Training Loop.

This is the same pattern whether you train a 131-parameter toy model or a 70B-parameter LLM:

```
for each epoch:
    Step 1 → logits = model(X)           forward pass
    Step 2 → loss = loss_fn(logits, y)   measure the error
    Step 3 → optimizer.zero_grad()   ⚠️  clear old gradients
    Step 4 → loss.backward()             compute new gradients
    Step 5 → optimizer.step()            update every weight
```

**Why `zero_grad()` on Step 3?**  
PyTorch *accumulates* gradients by default — it adds new gradients on top of old ones. Forget this line and the gradients from epoch 1 are still there in epoch 2, then in epoch 3, etc. They keep growing until training diverges. This is the **single most common PyTorch mistake**.

**What should loss look like?**  
For a random 3-class model: loss ≈ `log(3)` ≈ 1.099. As training progresses, watch it fall toward zero.

---
⚠️ **Common Mistake — `loss` vs `loss.item()` when logging:**  
`loss` is a tensor attached to the entire computation graph. Storing it in a list keeps that graph alive in RAM. After 100 epochs you've silently leaked hundreds of megabytes. Always call `.item()` to extract a plain Python float before appending to a list.

In [ ]:
import math

# ── Loss function ─────────────────────────────────────────────────────────────
# CrossEntropyLoss is the standard choice for multi-class classification.
# It expects: raw logits (batch, num_classes)  +  integer labels (batch,)
# Internally it applies Softmax → that is why forward() must NOT add Softmax.
loss_fn = nn.CrossEntropyLoss()

# ── Optimizer ─────────────────────────────────────────────────────────────────
# Adam: adapts the learning rate per parameter. The safest first choice.
# lr=0.01: step size. Too large → loss oscillates. Too small → barely moves.
# model.parameters() gives Adam references to all weights and biases.
optimizer = optim.Adam(model.parameters(), lr=0.01)

# ── Initial loss sanity check ─────────────────────────────────────────────────
# Before any training, a randomly initialised model on 3 classes
# should give loss ≈ log(3) ≈ 1.099.
# If it's wildly different, your data pipeline already has a bug.
model.eval()
with torch.no_grad():
    init_loss = loss_fn(model(X_train_t.to(device)), y_train_t.to(device))
print(f"Initial loss: {init_loss.item():.4f}   (expected ≈ {math.log(3):.4f})")
print()

In [ ]:
# ── The 5-Step Training Loop ──────────────────────────────────────────────────
EPOCHS = 100
train_losses = []   # Collect for plotting — use .item() not the tensor itself!

for epoch in range(1, EPOCHS + 1):

    model.train()   # Switch to training mode.
    # For this simple model it has no effect, but it's the correct habit:
    # models with Dropout or BatchNorm behave differently in train vs eval.

    # Step 1: Forward pass ────────────────────────────────────────────────────
    # Run all 120 training samples through the network.
    # (For large datasets you'd loop over mini-batches — same 5 steps inside)
    logits = model(X_train_t.to(device))              # shape: (120, 3)

    # Step 2: Compute loss ────────────────────────────────────────────────────
    loss = loss_fn(logits, y_train_t.to(device))      # scalar tensor

    # Step 3: Zero old gradients ──────────────────────────────────────────────
    # PyTorch ACCUMULATES gradients. Without this, gradients from the
    # previous epoch are still in .grad and get added to the new ones.
    # Result: effective gradient grows every epoch → training explodes.
    # This is the #1 most common PyTorch bug. Never skip this line.
    optimizer.zero_grad()

    # Step 4: Backpropagation ─────────────────────────────────────────────────
    # Computes ∂loss/∂(every weight) using the chain rule — automatically.
    # Each parameter's .grad attribute is filled in after this call.
    loss.backward()

    # Step 5: Update weights ──────────────────────────────────────────────────
    # Adam reads each parameter's .grad and nudges the weight
    # in the direction that reduces the loss.
    optimizer.step()

    # Track — use .item() to get a plain float, NOT the tensor
    # Storing the tensor keeps the computation graph alive → memory leak
    train_losses.append(loss.item())

    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d}/{EPOCHS}  |  Loss: {loss.item():.4f}")

print("\n✅ Training complete!")

In [ ]:
# ── Plot the learning curve ───────────────────────────────────────────────────
# This is your model's vital-signs chart. Learn to read it:
#
#   ✅  Loss steadily going down       → model is learning correctly
#   ❌  Loss barely moves              → learning rate too small, or data not scaled
#   ❌  Loss oscillates up and down    → learning rate too large
#   ❌  Loss shoots up toward infinity → learning rate way too large (diverging)

plt.figure(figsize=(8, 4))
plt.plot(train_losses, color='royalblue', lw=2, label='Training Loss')
plt.axhline(y=math.log(3), color='tomato', linestyle='--', alpha=0.6,
            label=f'Random baseline (log 3 ≈ {math.log(3):.2f})')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss Curve')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## ✅ Step 6 — Evaluate on Test Data

**What:** Measure how well the model performs on data it has never seen.

**Why evaluate separately?**  
Training loss tells you how well the model *memorized* the training set. Test accuracy tells you whether it *generalizes*. A model can reach 100% training accuracy while failing on new data — this is called **overfitting**.

**Two critical switches for evaluation:**

| Code | What it does |
|------|-------------|
| `model.eval()` | Disables Dropout; switches BatchNorm to use stored running stats |
| `torch.no_grad()` | Stops building the computation graph → faster and uses less memory |

---
⚠️ **Common Mistake — Evaluating without `model.eval()`:**  
If you have Dropout in your model and skip `model.eval()`, neurons are randomly dropped during evaluation. Your accuracy will be lower than the true value *and change every run* — a silent, confusing bug.

In [ ]:
# ── Switch to evaluation mode ─────────────────────────────────────────────────
model.eval()

with torch.no_grad():   # No gradients needed — saves memory and time
    test_logits = model(X_test_t.to(device))        # (30, 3) raw scores
    predictions = torch.argmax(test_logits, dim=1)  # pick class with highest score

predictions_np = predictions.cpu().numpy()

correct  = (predictions_np == y_test).sum()
accuracy = correct / len(y_test)

print(f"Test Accuracy: {correct}/{len(y_test)} = {accuracy:.1%}")
print()

# ── Per-class accuracy ────────────────────────────────────────────────────────
# Overall accuracy can hide per-class failures. Always break it down.
# Example: 93% overall might mean 100% on Setosa but only 80% on Virginica.
print("Per-class accuracy:")
for i, name in enumerate(CLASS_NAMES):
    mask    = (y_test == i)
    cls_acc = (predictions_np[mask] == i).mean()
    print(f"  {name:<12} : {cls_acc:.0%}  ({mask.sum()} test samples)")

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────────────────
# Shows WHICH classes the model confuses with each other.
#
# How to read it:
#   Each ROW = the true class
#   Each COLUMN = what the model predicted
#   Diagonal = correct predictions (want these to be large)
#   Off-diagonal = errors (want these to be 0)

cm = confusion_matrix(y_test, predictions_np)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=1, linecolor='white')
plt.title('Confusion Matrix — Test Set')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

---
## ✅ Step 7 — Predict on New Data

**What:** Use the trained model to classify a brand-new flower.

**Why a dedicated predict function?**  
Inference has different requirements than training:
1. Input comes from outside (not a pre-split tensor)
2. Must apply the **same scaler** used in training
3. Must use `eval()` + `no_grad()`
4. Should return a human-readable class name, not a class index

---
⚠️ **The Most Dangerous Production Bug — Not scaling inference data:**  
Your model was trained on scaled features (mean≈0, std≈1). If you feed raw centimeter measurements at inference time, the inputs are completely foreign to the model — it will give wrong predictions with full confidence, and no error message.  
**Rule: whatever transformation you applied to training data, you must apply to every new input using the exact same fitted scaler object.**

In [ ]:
def predict(raw_features):
    """
    Predict iris species from raw (unscaled) measurements.

    Args:
        raw_features : list of 4 floats
                       [sepal_length, sepal_width, petal_length, petal_width]

    Returns:
        (class_name, confidence, all_probabilities)
    """
    model.eval()

    # Step 1: Scale with the SAME scaler fitted on training data
    # This is the most important step — easy to forget, hard to debug.
    scaled = scaler.transform([raw_features])              # numpy (1, 4)
    tensor = torch.tensor(scaled, dtype=torch.float32).to(device)

    with torch.no_grad():
        logits = model(tensor)                             # (1, 3) raw scores

        # Softmax converts raw scores to probabilities that sum to 1.
        # We apply it HERE for output, not in forward().
        probs  = torch.softmax(logits, dim=1)              # (1, 3)

        pred_idx   = probs.argmax(dim=1).item()            # 0, 1, or 2
        confidence = probs[0, pred_idx].item()             # probability of top class

    return CLASS_NAMES[pred_idx], confidence, probs[0].cpu().numpy()


# ── Test on known samples ─────────────────────────────────────────────────────
test_flowers = [
    ([5.1, 3.5, 1.4, 0.2], 'setosa'),      # classic small petals
    ([6.7, 3.0, 5.2, 2.3], 'virginica'),   # classic large petals
    ([5.8, 2.7, 4.1, 1.0], 'versicolor'), # borderline case
]

print(f"{'Measurements':>30}  {'True':>12}  {'Predicted':>12}  {'Confidence':>11}")
print("-" * 72)
for features, true_label in test_flowers:
    name, conf, _ = predict(features)
    mark = '✅' if name == true_label else '❌'
    print(f"{str(features):>30}  {true_label:>12}  {name:>12}  {conf:>10.1%}  {mark}")

# ── Show full probability breakdown for the borderline case ──────────────────
print("\nProbability breakdown for [5.8, 2.7, 4.1, 1.0] (borderline):")
_, _, probs = predict([5.8, 2.7, 4.1, 1.0])
for name, p in zip(CLASS_NAMES, probs):
    bar = '█' * int(p * 30)
    print(f"  {name:<12}: {p:>5.1%}  {bar}")

---
## ✅ Step 8 — Save and Load the Model

**What:** Persist trained weights to disk and reload them later.

**The right way — `state_dict`:**  
A `state_dict` is a dictionary mapping layer names to their weight tensors. It is just numbers — no code. Small, portable, version-safe.

```
Save:  torch.save(model.state_dict(), 'model.pth')
Load:  model.load_state_dict(torch.load('model.pth'))
```

---
⚠️ **Common Mistake #1 — Forgetting `model.eval()` after loading:**  
After `load_state_dict`, the model is in training mode by default. Dropout is active. Always call `model.eval()` immediately after loading before running any inference.

⚠️ **Common Mistake #2 — Using `torch.save(model, ...)` instead of `state_dict`:**  
Saving the whole model uses Python `pickle` and ties the file to your exact class name and file path. If you rename the class or move the file, loading breaks. `state_dict` saves only the numbers — no code dependency.

In [ ]:
# ── Save ──────────────────────────────────────────────────────────────────────
torch.save(model.state_dict(), 'iris_model.pth')
print("Saved to iris_model.pth")

# Peek inside — just a dict of {name: weight_tensor}
print("\nstate_dict contents:")
for name, tensor in model.state_dict().items():
    print(f"  {name:<20}  shape: {tuple(tensor.shape)}")

# ── Load ──────────────────────────────────────────────────────────────────────
# Step 1: Create a fresh model with the same architecture
loaded_model = IrisNet().to(device)

# Step 2: Fill it with the saved weights
# map_location handles cross-device loading (saved on GPU, loading on CPU)
loaded_model.load_state_dict(torch.load('iris_model.pth', map_location=device))

# Step 3: ALWAYS set eval mode after loading before running inference
loaded_model.eval()
print("\nModel loaded successfully")

# ── Verify outputs are identical ──────────────────────────────────────────────
with torch.no_grad():
    sample = torch.tensor(
        scaler.transform([[5.1, 3.5, 1.4, 0.2]]), dtype=torch.float32
    ).to(device)
    original_out = model(sample)
    loaded_out   = loaded_model(sample)

match = torch.allclose(original_out, loaded_out)
print(f"Outputs match: {'✅ Yes' if match else '❌ No — something went wrong'}")

---
## 🏁 What You Built — The Full Picture

```
Raw CSV / Data
      │
      ├── train_test_split()    split FIRST, before scaling
      ├── StandardScaler        fit on TRAIN only → no data leakage
      └── torch.tensor()        float32 for X, int64 (long) for y
               │
           IrisNet
           Linear(4→16) → ReLU → Linear(16→3)
               │
        Training Loop (×100 epochs)
        forward → loss → zero_grad → backward → step
               │
        Evaluation   model.eval() + no_grad()
        Inference    scale with SAME scaler, then model.eval()
        Save/Load    state_dict only
```

---

## ⚠️ The 6 Mistakes to Never Make Again

| # | Mistake | What goes wrong | Fix |
|---|---------|----------------|-----|
| 1 | Forgetting `optimizer.zero_grad()` | Gradients accumulate → training diverges | Always Step 3 before Step 4 |
| 2 | Softmax in `forward()` + CrossEntropyLoss | Silent double-softmax, lower accuracy | Return raw logits from `forward()` |
| 3 | `scaler.fit_transform(X_test)` | Data leakage → inflated test metrics | `fit` train only, `transform` test |
| 4 | Not scaling inference inputs | Model sees foreign data → garbage output | Always transform with the same scaler |
| 5 | No `model.eval()` during testing | Dropout active → non-deterministic metrics | Always switch modes |
| 6 | Appending `loss` not `loss.item()` | Memory leak — computation graph kept alive per epoch | Always `.item()` when logging |

---

## 🚀 What to Try Next

1. **Change architecture** — add a third hidden layer (`Linear(16, 8)`). Does accuracy improve?
2. **Remove ReLU** — retrain. What happens? Why is it flat?
3. **Remove scaling** — retrain without `StandardScaler`. Compare the loss curves.
4. **Different dataset** — replace Iris with `load_wine()` from sklearn. Only change `nn.Linear(4, 16)` → `nn.Linear(13, 16)` and `num_classes=3`.
5. **Mini-batching** — wrap data in a `DataLoader` with `batch_size=16` and loop over it inside the training loop. Same 5 steps.